In [1]:
from sklearn.preprocessing import StandardScaler
import joblib
import os
from pathlib import Path
from pyspark.sql import functions as F

In [ ]:
ambiente = 'dev'
costa = 'Matamoros'

PORCENTAJE_SAMPLE_DATA = 0.7
FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_steepness'
]
RANDOM_SEED = 0

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
else:
    scaler_path = f'{Path.cwd().parent}/scaler/scaler_{{}}.pkl'

In [ ]:
data = (
    spark.sql(
        f"""
            SELECT coast_name, datetime, {', '.join(FEATURES)},
            CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
            FROM cor_{ambiente}.silver.swell_metrics
            WHERE coast_name = '{costa}'
        """
    )
)

In [ ]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}

data_sample = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
).toPandas()

In [5]:
scaler = StandardScaler()

In [ ]:
X = data_sample[FEATURES]
scaler.fit(X)

joblib.dump(scaler, scaler_path.format(costa))